# Model Experiments - Consumer Complaint Classification

This notebook contains experiments with different models and hyperparameters.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GridSearchCV, cross_val_score
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

## Load Processed Data

In [ ]:
# Load preprocessed data
train_df = pd.read_csv('../data/processed/train.csv')
val_df = pd.read_csv('../data/processed/val.csv')
test_df = pd.read_csv('../data/processed/test.csv')

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

## Feature Extraction Experiments

In [ ]:
from src.feature_engineering import FeatureExtractor

# Experiment with different feature extraction parameters
feature_configs = [
    {'method': 'tfidf', 'max_features': 5000, 'ngram_range': (1, 1)},
    {'method': 'tfidf', 'max_features': 5000, 'ngram_range': (1, 2)},
    {'method': 'tfidf', 'max_features': 10000, 'ngram_range': (1, 2)},
]

for config in feature_configs:
    print(f"\nConfig: {config}")
    
    extractor = FeatureExtractor(**config)
    X_train = extractor.fit_transform(train_df['processed_text'])
    
    print(f"Feature matrix shape: {X_train.shape}")
    print(f"Sparsity: {(1 - X_train.nnz / (X_train.shape[0] * X_train.shape[1])):.4f}")

## Hyperparameter Tuning

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

# Prepare features
extractor = FeatureExtractor(method='tfidf', max_features=5000, ngram_range=(1, 2))
X_train = extractor.fit_transform(train_df['processed_text'])
y_train = train_df['label_encoded'].values

# Grid search for Logistic Regression
param_grid = {
    'C': [0.1, 1.0, 10.0],
    'solver': ['lbfgs', 'saga'],
    'max_iter': [500, 1000]
}

grid_search = GridSearchCV(
    LogisticRegression(multi_class='multinomial', random_state=42),
    param_grid,
    cv=3,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best F1 score: {grid_search.best_score_:.4f}")

## Cross-Validation Comparison

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

# Define models
models = {
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, multi_class='multinomial'),
    'Naive Bayes': MultinomialNB(),
    'SVM': LinearSVC(C=1.0, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

# Cross-validate each model
cv_results = {}

for name, model in models.items():
    print(f"\nCross-validating {name}...")
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='f1_weighted', n_jobs=-1)
    cv_results[name] = scores
    print(f"F1 Score: {scores.mean():.4f} (+/- {scores.std():.4f})")

# Plot results
plt.figure(figsize=(12, 6))
plt.boxplot(cv_results.values(), labels=cv_results.keys())
plt.ylabel('F1 Score')
plt.title('Cross-Validation Results Comparison')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Feature Importance Analysis

In [ ]:
# Train model
lr_model = LogisticRegression(C=1.0, max_iter=1000, multi_class='multinomial')
lr_model.fit(X_train, y_train)

# Get feature names
feature_names = extractor.get_feature_names()

# Plot top features for each class
category_names = {
    0: "Credit reporting",
    1: "Debt collection",
    2: "Consumer Loan",
    3: "Mortgage"
}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

for idx, (cat_id, cat_name) in enumerate(category_names.items()):
    # Get coefficients for this class
    coef = lr_model.coef_[cat_id]
    top_indices = np.argsort(coef)[::-1][:15]
    
    top_features = [feature_names[i] for i in top_indices]
    top_coefs = [coef[i] for i in top_indices]
    
    # Plot
    y_pos = np.arange(len(top_features))
    axes[idx].barh(y_pos, top_coefs, color='steelblue')
    axes[idx].set_yticks(y_pos)
    axes[idx].set_yticklabels(top_features)
    axes[idx].invert_yaxis()
    axes[idx].set_xlabel('Coefficient')
    axes[idx].set_title(f'Top Features - {cat_name}', fontweight='bold')
    axes[idx].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/feature_importance.png', dpi=300)
plt.show()

## Learning Curves

In [ ]:
from sklearn.model_selection import learning_curve

# Calculate learning curves
train_sizes, train_scores, val_scores = learning_curve(
    LogisticRegression(C=1.0, max_iter=1000, multi_class='multinomial'),
    X_train, y_train,
    cv=3,
    n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='f1_weighted'
)

# Plot
train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_mean = val_scores.mean(axis=1)
val_std = val_scores.std(axis=1)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_mean, label='Training score', color='blue', marker='o')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color='blue')
plt.plot(train_sizes, val_mean, label='Cross-validation score', color='green', marker='s')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color='green')
plt.xlabel('Training Set Size')
plt.ylabel('F1 Score')
plt.title('Learning Curves - Logistic Regression')
plt.legend(loc='best')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../results/figures/learning_curves.png', dpi=300)
plt.show()